# 01 — CNN dari Nol: Klasifikasi CIFAR-10

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riki-profile/ai-engineer-journey/blob/main/02-cnn/01_cnn_dari_nol.ipynb)

Di modul 01 kita memakai MLP: gambar di-*flatten* menjadi vektor 784 angka. Cara itu membuang informasi **spasial** (piksel mana bertetangga dengan piksel mana). **Convolutional Neural Network (CNN)** mempertahankan struktur 2D gambar dan jauh lebih efisien untuk data visual.

Isi notebook:
1. Cara kerja **konvolusi** dan **pooling**, termasuk rumus ukuran output.
2. Melacak **ukuran output tiap layer** di dalam CNN.
3. **Data augmentation** untuk mengurangi overfitting.
4. Melatih CNN sederhana di **CIFAR-10** (60.000 gambar berwarna 32×32, 10 kelas).

> Notebook ini dirancang untuk **Google Colab dengan GPU**.

## 0. Cek GPU

Jalankan sel ini paling awal. Jika GPU belum aktif, di Colab pilih **Runtime → Change runtime type → T4 GPU**, lalu jalankan ulang dari atas.

Notebook tetap bisa berjalan di CPU, tetapi training akan jauh lebih lambat. Untuk uji cepat di CPU, gunakan `QUICK_RUN` (lihat bagian konfigurasi).

In [ ]:
import shutil
import subprocess
import torch

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"✅ GPU terdeteksi: {props.name} | memori {props.total_memory / 1e9:.1f} GB")
else:
    print("⚠️ GPU tidak terdeteksi, training akan berjalan di CPU (lambat).")
    print("   Di Colab: Runtime → Change runtime type → T4 GPU")

# nvidia-smi menampilkan detail GPU & pemakaian memori (hanya ada jika driver NVIDIA terpasang)
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"])

## 1. Setup dan Konfigurasi

Import library, set seed, dan tentukan device.

`QUICK_RUN` dipakai untuk uji cepat (subset sangat kecil dan 1 epoch). Nilainya dibaca dari environment variable `QUICK_RUN=1`; secara default notebook memakai dataset penuh.

In [ ]:
import os
import time
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

QUICK_RUN = os.environ.get("QUICK_RUN") == "1"
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
EPOCHS = 1 if QUICK_RUN else 15
DATA_DIR = "./data"
NUM_WORKERS = 2  # jumlah proses paralel untuk memuat data (Colab punya 2 core CPU)

print("device:", device, "| QUICK_RUN:", QUICK_RUN, "| EPOCHS:", EPOCHS)

## 2. Memuat CIFAR-10

Kita unduh data train sekali (`download=True`), lalu intip beberapa gambar. Atribut `.data` berisi array NumPy mentah berbentuk `(N, 32, 32, 3)` dengan nilai 0–255.

In [ ]:
cifar_train_raw = datasets.CIFAR10(DATA_DIR, train=True, download=True)
CLASS_NAMES = cifar_train_raw.classes
print("Jumlah gambar train:", len(cifar_train_raw))
print("Bentuk array mentah:", cifar_train_raw.data.shape)
print("Kelas:", CLASS_NAMES)

fig, axes = plt.subplots(2, 8, figsize=(12, 3.5))
for i, ax in enumerate(axes.flat):
    ax.imshow(cifar_train_raw.data[i])
    ax.set_title(CLASS_NAMES[cifar_train_raw.targets[i]], fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 3. Konvolusi

### 3.1 Intuisi

Konvolusi menggeser sebuah **kernel** (filter kecil, misalnya 3×3) di atas gambar. Di setiap posisi, elemen kernel dikalikan dengan piksel di bawahnya lalu dijumlahkan, menghasilkan satu angka di **feature map** output.

Keunggulan dibanding layer `Linear`:
- **Parameter sedikit**: kernel 3×3 yang sama dipakai di seluruh posisi gambar (*weight sharing*).
- **Lokalitas**: setiap output hanya melihat area kecil di sekitarnya, cocok untuk mendeteksi pola lokal seperti tepi dan tekstur.

Contoh di bawah memakai kernel **Sobel** buatan tangan untuk mendeteksi tepi vertikal dan horizontal. Pada CNN, nilai kernel seperti ini tidak ditulis manual, melainkan **dipelajari** saat training.

In [ ]:
# Ambil satu gambar, ubah ke grayscale, bentuk (N=1, C=1, H=32, W=32)
img = torch.tensor(cifar_train_raw.data[7], dtype=torch.float32).mean(dim=2) / 255.0
img = img.unsqueeze(0).unsqueeze(0)

sobel_x = torch.tensor([[-1., 0., 1.],
                        [-2., 0., 2.],
                        [-1., 0., 1.]])   # peka terhadap tepi vertikal
sobel_y = sobel_x.T                         # peka terhadap tepi horizontal
# Bentuk kernel untuk conv2d: (out_channels, in_channels, kH, kW)
kernels = torch.stack([sobel_x, sobel_y]).unsqueeze(1)

edges = F.conv2d(img, kernels, padding=1)
print("input :", tuple(img.shape))
print("kernel:", tuple(kernels.shape))
print("output:", tuple(edges.shape), "→ 2 feature map, satu per kernel")

fig, ax = plt.subplots(1, 3, figsize=(9, 3))
for a, data, title in zip(ax, [img[0, 0], edges[0, 0].abs(), edges[0, 1].abs()],
                          ["asli (grayscale)", "tepi vertikal", "tepi horizontal"]):
    a.imshow(data, cmap="gray"); a.set_title(title); a.axis("off")
plt.tight_layout()
plt.show()

### 3.2 `nn.Conv2d` dan jumlah parameternya

`nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding)`:
- `in_channels`: jumlah channel input (3 untuk RGB).
- `out_channels`: jumlah filter = jumlah feature map output.
- Setiap filter berukuran `in_channels × k × k`, ditambah 1 bias.

Jumlah parameter = `(in_channels × k × k + 1) × out_channels`. Perhatikan: jumlah ini **tidak bergantung pada ukuran gambar**.

In [ ]:
conv = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
x = torch.randn(8, 3, 32, 32)   # batch 8 gambar RGB 32×32
print("input :", tuple(x.shape))
print("output:", tuple(conv(x).shape))
print("bobot :", tuple(conv.weight.shape), "| bias:", tuple(conv.bias.shape))

n_param = sum(p.numel() for p in conv.parameters())
print(f"parameter: {n_param} = (3×3×3 + 1) × 16 = {(3*3*3 + 1) * 16}")

# Bandingkan: Linear yang menghubungkan gambar 3×32×32 ke 16×32×32 output
print(f"Linear setara butuh: {(3*32*32) * (16*32*32):,} bobot!")

### 3.3 Rumus ukuran output

Untuk input berukuran $W$, kernel $K$, padding $P$, dan stride $S$:

$$W_{out} = \left\lfloor \frac{W - K + 2P}{S} \right\rfloor + 1$$

- **Padding** menambahkan piksel nol di tepi. Dengan $K=3, P=1, S=1$, ukuran tetap sama ("same padding").
- **Stride** adalah besar langkah geser kernel. Stride 2 membuat ukuran kira-kira setengahnya.

Mari cocokkan rumus dengan hasil PyTorch untuk beberapa konfigurasi.

In [ ]:
def conv_out_size(w, k, p=0, s=1):
    # Rumus ukuran output konvolusi / pooling
    return (w - k + 2 * p) // s + 1

x = torch.randn(1, 3, 32, 32)
configs = [(3, 0, 1), (3, 1, 1), (5, 2, 1), (3, 1, 2), (7, 3, 2)]
print(f"{'kernel':>6} {'pad':>4} {'stride':>6} | {'rumus':>5} {'pytorch':>7}")
for k, p, s in configs:
    out = nn.Conv2d(3, 8, kernel_size=k, padding=p, stride=s)(x)
    print(f"{k:>6} {p:>4} {s:>6} | {conv_out_size(32, k, p, s):>5} {out.shape[-1]:>7}")

## 4. Pooling

**Max pooling** mengambil nilai maksimum di setiap jendela (umumnya 2×2 dengan stride 2), sehingga ukuran spasial menjadi setengah. Manfaatnya:
- Mengurangi komputasi di layer berikutnya.
- Memberi sedikit toleransi terhadap pergeseran posisi objek.
- Tidak punya parameter yang dipelajari.

Rumus ukuran output-nya sama dengan konvolusi.

In [ ]:
x = torch.tensor([[1., 3., 2., 1.],
                  [4., 6., 5., 0.],
                  [7., 2., 9., 8.],
                  [1., 0., 3., 4.]]).reshape(1, 1, 4, 4)

print("input 4×4:\n", x[0, 0])
print("max pool 2×2:\n", nn.MaxPool2d(2)(x)[0, 0])
print("avg pool 2×2:\n", nn.AvgPool2d(2)(x)[0, 0])

## 5. Data Augmentation

**Data augmentation** membuat variasi acak dari gambar train setiap kali gambar diambil (misalnya dibalik atau digeser sedikit). Model jadi "melihat" data yang sedikit berbeda setiap epoch, sehingga lebih sulit menghafal dan **overfitting berkurang**.

Aturan penting:
- Augmentasi **hanya untuk data train**.
- Data validasi/test hanya dinormalisasi agar evaluasinya konsisten.

Augmentasi yang dipakai:
- `RandomCrop(32, padding=4)`: tambah padding 4 piksel lalu crop acak 32×32 (efeknya menggeser gambar).
- `RandomHorizontalFlip()`: membalik gambar kiri-kanan dengan peluang 50%.

Nilai `mean` dan `std` adalah statistik per channel (R, G, B) dari data train CIFAR-10.

In [ ]:
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])
eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

# Visualisasi: satu gambar diaugmentasi 7 kali (tanpa ToTensor/Normalize agar mudah ditampilkan)
aug_only = transforms.Compose([transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip()])
contoh = Image.fromarray(cifar_train_raw.data[3])

fig, axes = plt.subplots(1, 8, figsize=(12, 2))
axes[0].imshow(contoh); axes[0].set_title("asli", fontsize=9)
for ax in axes[1:]:
    ax.imshow(aug_only(contoh)); ax.set_title("augmentasi", fontsize=9)
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

### 5.1 Split train / validasi dengan transform berbeda

Masalah kecil: jika kita memakai `random_split` pada satu dataset, data validasi ikut ter-augmentasi. Solusinya: buat **dua objek dataset** dari file yang sama (satu dengan `train_transform`, satu dengan `eval_transform`), lalu ambil indeks yang tidak saling tumpang tindih dengan `Subset`.

Pembagian: 45.000 train, 5.000 validasi, dan 10.000 test (dari file test terpisah).

In [ ]:
train_full = datasets.CIFAR10(DATA_DIR, train=True, download=True, transform=train_transform)
train_full_eval = datasets.CIFAR10(DATA_DIR, train=True, download=True, transform=eval_transform)
test_full = datasets.CIFAR10(DATA_DIR, train=False, download=True, transform=eval_transform)

indices = torch.randperm(len(train_full), generator=torch.Generator().manual_seed(42)).tolist()
val_idx, train_idx = indices[:5000], indices[5000:]
test_idx = list(range(len(test_full)))

if QUICK_RUN:
    # Subset sangat kecil untuk uji cepat di CPU
    train_idx, val_idx, test_idx = train_idx[:512], val_idx[:128], test_idx[:128]

train_set = Subset(train_full, train_idx)
val_set = Subset(train_full_eval, val_idx)
test_set = Subset(test_full, test_idx)

loader_args = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=(device == "cuda"))
train_loader = DataLoader(train_set, shuffle=True, **loader_args)
val_loader = DataLoader(val_set, shuffle=False, **loader_args)
test_loader = DataLoader(test_set, shuffle=False, **loader_args)

print(f"train: {len(train_set)} | val: {len(val_set)} | test: {len(test_set)}")

## 6. Arsitektur CNN

Model terdiri dari 3 **blok konvolusi** yang polanya sama:

`Conv 3×3 → BatchNorm → ReLU → Conv 3×3 → BatchNorm → ReLU → MaxPool 2×2`

- Setiap blok **menggandakan jumlah channel** (32 → 64 → 128) dan **membagi dua ukuran spasial** (32 → 16 → 8 → 4). Semakin dalam, fitur makin abstrak (tepi → tekstur → bagian objek).
- **BatchNorm** menormalkan output tiap layer per batch, sehingga training lebih stabil dan cepat. Karena BatchNorm sudah punya parameter geser (β), conv sebelumnya tidak perlu bias (`bias=False`).
- Bagian `classifier` di akhir mirip MLP dari modul 01.

Model dibagi menjadi `features` dan `classifier` agar mudah dilacak ukuran output-nya.

In [ ]:
def conv_block(in_ch, out_ch):
    # Dua konvolusi 3×3 (ukuran tetap) lalu max pool (ukuran jadi setengah)
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(2),
    )


class SimpleCNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            conv_block(3, 32),     # 3×32×32  -> 32×16×16
            conv_block(32, 64),    # 32×16×16 -> 64×8×8
            conv_block(64, 128),   # 64×8×8   -> 128×4×4
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),                 # 128×4×4 -> 2048
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, n_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


model = SimpleCNN().to(device)
print(f"Jumlah parameter: {sum(p.numel() for p in model.parameters()):,}")

### 6.1 Melacak ukuran output tiap layer

Kesalahan paling umum saat membuat CNN adalah **ukuran yang tidak cocok** antara bagian konvolusi dan layer `Linear` pertama. Cara mengeceknya: kirim tensor *dummy* melewati setiap layer dan cetak bentuknya.

Perhatikan pola: Conv (padding 1) mempertahankan ukuran, MaxPool membaginya dua, dan Flatten mengubah `128×4×4` menjadi `2048` sehingga cocok dengan `nn.Linear(128 * 4 * 4, 256)`.

In [ ]:
x = torch.randn(1, 3, 32, 32, device=device)
print(f"{'layer':<14} {'output shape':<22} {'parameter':>9}")
print(f"{'input':<14} {str(tuple(x.shape)):<22}")

model.eval()  # BatchNorm butuh mode eval untuk batch berukuran 1
with torch.no_grad():
    for bagian in [model.features, model.classifier]:
        for layer in bagian.modules():
            # Lewati container (Sequential), hanya proses layer "daun"
            if list(layer.children()):
                continue
            x = layer(x)
            n_p = sum(p.numel() for p in layer.parameters())
            print(f"{layer.__class__.__name__:<14} {str(tuple(x.shape)):<22} {n_p:>9,}")

## 7. Loss, Optimizer, dan Fungsi Training

Fungsi `train_one_epoch` dan `evaluate` sama persis dengan modul 01: *forward → loss → backward → step → zero_grad* untuk training, dan `model.eval()` + `torch.no_grad()` untuk evaluasi.

Tambahan baru: **learning rate scheduler** `CosineAnnealingLR`, yang menurunkan learning rate secara bertahap mengikuti kurva cosinus. Langkah besar di awal membuat belajar cepat, langkah kecil di akhir membantu model "mengendap" di solusi yang baik.

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)


def train_one_epoch(model, loader, loss_fn, optimizer):
    model.train()
    total_loss, total_benar, total_data = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = loss_fn(logits, y)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        total_loss += loss.item() * x.size(0)
        total_benar += (logits.argmax(dim=1) == y).sum().item()
        total_data += x.size(0)
    return total_loss / total_data, total_benar / total_data


@torch.no_grad()
def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss, total_benar, total_data = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        total_loss += loss_fn(logits, y).item() * x.size(0)
        total_benar += (logits.argmax(dim=1) == y).sum().item()
        total_data += x.size(0)
    return total_loss / total_data, total_benar / total_data

## 8. Training

Dengan GPU T4 di Colab, satu epoch penuh butuh sekitar 15–30 detik. `scheduler.step()` dipanggil sekali per epoch, setelah training epoch tersebut selesai.

In [ ]:
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
mulai = time.time()

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, loss_fn, optimizer)
    val_loss, val_acc = evaluate(model, val_loader, loss_fn)
    scheduler.step()

    for k, v in zip(history, [train_loss, train_acc, val_loss, val_acc]):
        history[k].append(v)
    print(f"Epoch {epoch:2d}/{EPOCHS} | train loss {train_loss:.4f} acc {train_acc:.3f} | "
          f"val loss {val_loss:.4f} acc {val_acc:.3f} | lr {scheduler.get_last_lr()[0]:.1e}")

print(f"\nTotal waktu training: {time.time() - mulai:.0f} detik")

## 9. Grafik Loss dan Akurasi

Berkat augmentation, jarak antara kurva train dan validasi biasanya tetap kecil. Coba bandingkan dengan training tanpa augmentation di bagian Latihan.

In [ ]:
epochs_range = range(1, EPOCHS + 1)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for a, key, judul in [(ax[0], "loss", "Loss"), (ax[1], "acc", "Akurasi")]:
    a.plot(epochs_range, history[f"train_{key}"], "o-", label="train")
    a.plot(epochs_range, history[f"val_{key}"], "o-", label="validasi")
    a.set_xlabel("epoch"); a.set_title(judul); a.legend()
    a.set_xticks(list(epochs_range))
plt.tight_layout()
plt.show()

## 10. Evaluasi di Data Test dan Akurasi per Kelas

Akurasi total bisa menyembunyikan kelemahan di kelas tertentu. Dengan menghitung akurasi per kelas, kita bisa melihat kelas mana yang paling sulit (pada CIFAR-10 biasanya *cat* dan *dog*).

In [ ]:
test_loss, test_acc = evaluate(model, test_loader, loss_fn)
print(f"Test loss: {test_loss:.4f} | Test akurasi: {test_acc:.3f}\n")

# Kumpulkan semua prediksi di data test
model.eval()
semua_pred, semua_label = [], []
with torch.no_grad():
    for x, y in test_loader:
        semua_pred.append(model(x.to(device)).argmax(dim=1).cpu())
        semua_label.append(y)
semua_pred = torch.cat(semua_pred)
semua_label = torch.cat(semua_label)

for i, nama in enumerate(CLASS_NAMES):
    mask = semua_label == i
    if mask.sum() > 0:
        acc = (semua_pred[mask] == i).float().mean().item()
        print(f"{nama:<12} {acc:6.1%}  {'█' * int(acc * 30)}")

## 11. Contoh Prediksi

Gambar dikembalikan ke skala asli (*denormalize*) per channel sebelum ditampilkan. Judul **hijau** = benar, **merah** = salah.

In [ ]:
def denormalize(img):
    # Kebalikan Normalize per channel, lalu ubah (C, H, W) -> (H, W, C) untuk matplotlib
    mean = torch.tensor(CIFAR_MEAN).view(3, 1, 1)
    std = torch.tensor(CIFAR_STD).view(3, 1, 1)
    return (img * std + mean).clamp(0, 1).permute(1, 2, 0)

x_test, y_test = next(iter(test_loader))
with torch.no_grad():
    probs = torch.softmax(model(x_test.to(device)), dim=1).cpu()
preds = probs.argmax(dim=1)

fig, axes = plt.subplots(3, 6, figsize=(12, 7))
for i, ax in enumerate(axes.flat):
    ax.imshow(denormalize(x_test[i]))
    benar = preds[i] == y_test[i]
    ax.set_title(f"pred: {CLASS_NAMES[preds[i]]} ({probs[i, preds[i]]:.0%})\nasli: {CLASS_NAMES[y_test[i]]}",
                 fontsize=8, color="green" if benar else "red")
    ax.axis("off")
plt.tight_layout()
plt.show()

## Ringkasan

- Konvolusi memakai kernel kecil yang sama di seluruh gambar: parameter sedikit dan menangkap pola lokal.
- Ukuran output: $\lfloor (W - K + 2P)/S \rfloor + 1$. Conv 3×3 dengan padding 1 mempertahankan ukuran, MaxPool 2×2 membaginya dua.
- Selalu lacak bentuk tensor antar layer untuk menghindari *shape mismatch*.
- Augmentation hanya untuk data train dan efektif mengurangi overfitting.
- Dengan 15 epoch penuh, CNN ini biasanya mencapai akurasi test sekitar **85–88%**, jauh di atas MLP untuk data yang sama (sekitar 50–55%).

## Latihan

1. **Efek augmentation.** Latih ulang model yang sama **tanpa** augmentation (pakai `eval_transform` untuk data train). Plot kurva train vs validasi kedua percobaan berdampingan. Mana yang lebih overfitting, dan seberapa besar selisih akurasi test-nya?
2. **Hitung ukuran secara manual.** Rancang CNN baru untuk gambar **64×64** dengan susunan: `Conv(3→16, k=5, s=1, p=0)` → `MaxPool(2)` → `Conv(16→32, k=3, s=2, p=1)` → `MaxPool(2)` → `Flatten` → `Linear(?, 10)`. Hitung dulu dengan rumus di atas ukuran setiap layer dan nilai `?` di kertas, lalu verifikasi dengan kode pelacak ukuran di bagian 6.1. Hitung juga jumlah parameternya secara manual.
3. **Visualisasi filter & feature map.** Tampilkan 32 filter dari conv pertama model yang sudah dilatih (`model.features[0][0].weight`, normalisasi ke 0–1 lalu tampilkan sebagai gambar RGB 3×3). Kemudian pakai *forward hook* (`register_forward_hook`) untuk menampilkan 8 feature map dari tiap blok untuk satu gambar test. Apa perbedaan pola di blok awal dan blok akhir?